In [ ]:
from glob import glob
from sklearn.feature_extraction.text import CountVectorizer
import json
import os
import re
import itertools
import random
from tqdm import tqdm
from multiprocess import Pool
from collections import defaultdict

def generate_trigrams(text):
    words = text.split()
    return list(zip(words, words[1:], words[2:]))

def skip_trigrams(text):
    trigrams = generate_trigrams(text)
    count = defaultdict(int)
    total = 0
    for t in trigrams:
        count[''.join(t)] += 1
        total += 1
    if len(count.keys()) < 3:
        return True
    for k, v in count.items():
        if (v / total) > 0.2:
            return True
    return False

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [ ]:
import os
import json
import re
import random
from collections import defaultdict
from sklearn.feature_extraction.text import CountVectorizer

# daftar kalimat yang ditolak
rejected = [
    'terima kasih kerana menonton',
    'terima kasih',
    'thank you for watching',
]

def new_path(f):
    return f.replace('_processed/', '_processed_trim_moshi/').replace('.mp3', '.moshi')

def build_pairs_from_json(json_file):
    data = []

    try:
        with open(json_file) as fopen:
            d = json.load(fopen)

    except:
        print(f"Gagal load {json_file}")
        return []

    folder = os.path.split(json_file)[0]
    folder_folder = os.path.split(folder)[1]
    speakers = defaultdict(dict)

    # filtering text + mapping audio
    for no, obj in enumerate(d):
        text = obj["text"].strip()
        audio_path = obj["audio_path"]

        rt_ = re.sub('[^a-z ]+', '', text.lower()).strip()
        if any([s == rt_ for s in rejected]):
            continue

        split = text.split()
        ones = [w for w in split if len(w) <= 1]
        if (len(ones) / len(split)) >= 0.5:
            continue

        if any([(len(set(w)) / len(w)) < 0.3 for w in split]):
            continue

        try:
            dense = CountVectorizer(ngram_range=(3, 3)).fit_transform([text]).todense()
            repeat = (dense > 3).sum() >= 1
            if repeat:
                continue
        except:
            continue

        if not os.path.exists(audio_path):
            # print("❌ Tidak ada file audio:", audio_path)
            continue
        
        if not os.path.exists(new_path(audio_path)):
            # print("❌ Tidak ada file new_path:", new_path(audio_path))
            continue
        
        speakers[obj['speaker']][no] = {
            "audio": audio_path,
            "transcription": text,
        }

    # generate pasangan reference-target
    for speaker in speakers.keys():
        data_ = []
        for row in speakers[speaker]:
            for row_ in speakers[speaker]:
                if row == row_:
                    continue
                data_.append({
                    "reference_audio": speakers[speaker][row]["audio"],
                    "reference_text": speakers[speaker][row]["transcription"],
                    "target_audio": speakers[speaker][row_]["audio"],
                    "target_text": speakers[speaker][row_]["transcription"],
                })
        if data_:  # ambil max 30 sampel
            data.extend(random.sample(data_, min(len(data_), 30)))

    return data


In [ ]:
pairs = build_pairs_from_json("/home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/indspeech_news_tts.json")

In [ ]:
import pandas as pd

pd.DataFrame(all_pairs).to_parquet('permutate.parquet')

In [ ]:
from datasets import load_dataset

dataset = load_dataset("parquet", data_files={'train': 'permutate.parquet'})

In [ ]:
import IPython.display as ipd
from IPython.display import display

sample = dataset["train"][6000]

print("Reference text:", sample["reference_text"])
display(ipd.Audio(sample["reference_audio"]))

print("Target text:", sample["target_text"])
display(ipd.Audio(sample["target_audio"]))
